# Классификация аудио: Сравнение классических и глубоких методов

В этой работе мы:
1. Загрузим датасет для бинарной классификации птиц
2. Извлечём MFCC признаки и обучим классические модели
3. Используем предобученную модель (Wav2Vec2) для извлечения признаков
4. Обучим классификационную голову и дообучим всю модель
5. Сравним все подходы и визуализируем результаты

## 0. Установка зависимостей

In [ ]:
!pip install datasets librosa scikit-learn matplotlib seaborn tqdm transformers torch torchaudio umap-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.manifold import TSNE
from umap import UMAP

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from transformers import Wav2Vec2Model, Wav2Vec2Processor

# Проверка доступности GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используемое устройство: {device}")

## 1. Загрузка и подготовка данных

In [ ]:
# Загрузка датасета BirdNet для бинарной классификации
dataset = load_dataset("capa2000/binary-classifier-birdnet")
print(f"Структура датасета: {dataset}")
print(f"\nПример записи:")
print(dataset['train'][0])

In [ ]:
# Извлечение данных
train_data = dataset['train']

# Получаем аудио и метки
audios = []
labels = []
sample_rates = []

print("Загрузка аудиофайлов...")
for item in tqdm(train_data):
    audio_array = item['audio']['array']
    sr = item['audio']['sampling_rate']
    label = item['label']
    
    audios.append(audio_array)
    labels.append(label)
    sample_rates.append(sr)

labels = np.array(labels)
print(f"\nЗагружено {len(audios)} аудиозаписей")
print(f"Распределение классов: {np.bincount(labels)}")
print(f"Частота дискретизации: {sample_rates[0]} Hz")

In [ ]:
# Разбиение на тренировочную и тестовую выборки
indices = np.arange(len(audios))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42, stratify=labels)

train_audios = [audios[i] for i in train_idx]
test_audios = [audios[i] for i in test_idx]
train_labels = labels[train_idx]
test_labels = labels[test_idx]

# Сохраняем оригинальные sample_rate
original_sr = sample_rates[0]

print(f"Тренировочная выборка: {len(train_audios)} записей")
print(f"Тестовая выборка: {len(test_audios)} записей")
print(f"Распределение в train: {np.bincount(train_labels)}")
print(f"Распределение в test: {np.bincount(test_labels)}")

## 2. Извлечение MFCC признаков

In [ ]:
def extract_mfcc_features(audio, sr, n_mfcc=13, aggregate='mean_std'):
    """
    Извлекает MFCC признаки из аудио и агрегирует их в один вектор.
    
    Parameters:
    - audio: аудио сигнал
    - sr: частота дискретизации
    - n_mfcc: количество MFCC коэффициентов
    - aggregate: метод агрегации ('mean', 'mean_std', 'mean_std_delta')
    
    Returns:
    - features: агрегированный вектор признаков
    """
    # Извлечение MFCC (возвращает матрицу [n_mfcc x time_frames])
    mfcc = librosa.feature.mfcc(y=audio.astype(np.float32), sr=sr, n_mfcc=n_mfcc)
    
    if aggregate == 'mean':
        # Среднее по времени
        features = np.mean(mfcc, axis=1)
    
    elif aggregate == 'mean_std':
        # Среднее и стандартное отклонение по времени
        mean = np.mean(mfcc, axis=1)
        std = np.std(mfcc, axis=1)
        features = np.concatenate([mean, std])
    
    elif aggregate == 'mean_std_delta':
        # MFCC + дельта + дельта-дельта, затем статистики
        delta = librosa.feature.delta(mfcc)
        delta2 = librosa.feature.delta(mfcc, order=2)
        
        all_features = np.vstack([mfcc, delta, delta2])
        mean = np.mean(all_features, axis=1)
        std = np.std(all_features, axis=1)
        features = np.concatenate([mean, std])
    
    return features

In [ ]:
# Извлечение MFCC признаков для всех аудио
print("Извлечение MFCC признаков для тренировочной выборки...")
train_mfcc = np.array([extract_mfcc_features(audio, original_sr, aggregate='mean_std_delta') 
                       for audio in tqdm(train_audios)])

print("\nИзвлечение MFCC признаков для тестовой выборки...")
test_mfcc = np.array([extract_mfcc_features(audio, original_sr, aggregate='mean_std_delta') 
                      for audio in tqdm(test_audios)])

print(f"\nРазмерность признаков MFCC: {train_mfcc.shape[1]}")
print(f"Тренировочная матрица: {train_mfcc.shape}")
print(f"Тестовая матрица: {test_mfcc.shape}")

In [ ]:
# Нормализация признаков
scaler_mfcc = StandardScaler()
train_mfcc_scaled = scaler_mfcc.fit_transform(train_mfcc)
test_mfcc_scaled = scaler_mfcc.transform(test_mfcc)

## 3. Визуализация MFCC признаков (снижение размерности)

In [ ]:
def visualize_features(features, labels, title, method='tsne'):
    """
    Визуализирует признаки с использованием t-SNE или UMAP.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # t-SNE
    print("Вычисление t-SNE...")
    tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(features)-1))
    features_tsne = tsne.fit_transform(features)
    
    scatter1 = axes[0].scatter(features_tsne[:, 0], features_tsne[:, 1], 
                                c=labels, cmap='coolwarm', alpha=0.7, s=50)
    axes[0].set_title(f'{title} - t-SNE')
    axes[0].set_xlabel('t-SNE 1')
    axes[0].set_ylabel('t-SNE 2')
    plt.colorbar(scatter1, ax=axes[0], label='Класс')
    
    # UMAP
    print("Вычисление UMAP...")
    umap = UMAP(n_components=2, random_state=42)
    features_umap = umap.fit_transform(features)
    
    scatter2 = axes[1].scatter(features_umap[:, 0], features_umap[:, 1], 
                                c=labels, cmap='coolwarm', alpha=0.7, s=50)
    axes[1].set_title(f'{title} - UMAP')
    axes[1].set_xlabel('UMAP 1')
    axes[1].set_ylabel('UMAP 2')
    plt.colorbar(scatter2, ax=axes[1], label='Класс')
    
    plt.tight_layout()
    plt.show()
    
    return features_tsne, features_umap

In [ ]:
# Визуализация MFCC признаков
mfcc_tsne, mfcc_umap = visualize_features(train_mfcc_scaled, train_labels, 'MFCC признаки')

## 4. Классические модели машинного обучения на MFCC

In [ ]:
def evaluate_classifiers(X_train, X_test, y_train, y_test, feature_name=''):
    """
    Оценивает несколько классификаторов и возвращает результаты.
    """
    classifiers = {
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
        'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
        'SVM (RBF)': SVC(kernel='rbf', random_state=42)
    }
    
    results = {}
    
    for name, clf in classifiers.items():
        print(f"\nОбучение {name}...")
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        
        accuracy = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='weighted')
        
        results[name] = {
            'accuracy': accuracy,
            'f1_score': f1,
            'predictions': y_pred,
            'model': clf
        }
        
        print(f"  Accuracy: {accuracy:.4f}")
        print(f"  F1-Score: {f1:.4f}")
    
    # Сводная таблица
    print(f"\n{'='*50}")
    print(f"Сводка результатов для {feature_name}:")
    print(f"{'='*50}")
    results_df = pd.DataFrame({
        'Модель': list(results.keys()),
        'Accuracy': [r['accuracy'] for r in results.values()],
        'F1-Score': [r['f1_score'] for r in results.values()]
    }).sort_values('F1-Score', ascending=False)
    print(results_df.to_string(index=False))
    
    return results, results_df

In [ ]:
# Оценка классификаторов на MFCC признаках
mfcc_results, mfcc_results_df = evaluate_classifiers(
    train_mfcc_scaled, test_mfcc_scaled, 
    train_labels, test_labels,
    feature_name='MFCC'
)

In [ ]:
# Confusion matrix для лучшей модели
best_model_name = mfcc_results_df.iloc[0]['Модель']
best_predictions = mfcc_results[best_model_name]['predictions']

plt.figure(figsize=(8, 6))
cm = confusion_matrix(test_labels, best_predictions)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'Confusion Matrix - {best_model_name} (MFCC)')
plt.xlabel('Предсказанный класс')
plt.ylabel('Истинный класс')
plt.show()

## 5. Извлечение признаков с помощью Wav2Vec2

Wav2Vec2 - это предобученная модель, которая:
- Работает с сырым аудио (не со спектрограммой)
- Требует частоту дискретизации 16kHz
- Извлекает последовательность векторов (нужна агрегация)

In [ ]:
# Загрузка предобученной модели Wav2Vec2
print("Загрузка Wav2Vec2...")
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")
wav2vec_model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")
wav2vec_model = wav2vec_model.to(device)
wav2vec_model.eval()

print(f"Модель загружена на устройство: {device}")
print(f"Требуемая частота дискретизации: 16000 Hz")

In [ ]:
def extract_wav2vec_features(audio, sr, model, processor, target_sr=16000):
    """
    Извлекает признаки из аудио с помощью Wav2Vec2.
    
    Returns:
    - features: агрегированный вектор признаков
    """
    # Ресэмплинг если нужно
    if sr != target_sr:
        audio = librosa.resample(audio.astype(np.float32), orig_sr=sr, target_sr=target_sr)
    
    # Подготовка входа для модели
    inputs = processor(audio, sampling_rate=target_sr, return_tensors="pt", padding=True)
    input_values = inputs.input_values.to(device)
    
    # Извлечение признаков
    with torch.no_grad():
        outputs = model(input_values)
        # outputs.last_hidden_state имеет форму [batch, seq_len, hidden_size]
        hidden_states = outputs.last_hidden_state.squeeze(0)  # [seq_len, hidden_size]
    
    # Агрегация: среднее по временной оси
    features = hidden_states.mean(dim=0).cpu().numpy()
    
    return features

In [ ]:
# Извлечение Wav2Vec2 признаков
print("Извлечение Wav2Vec2 признаков для тренировочной выборки...")
train_wav2vec = []
for audio in tqdm(train_audios):
    features = extract_wav2vec_features(audio, original_sr, wav2vec_model, processor)
    train_wav2vec.append(features)
train_wav2vec = np.array(train_wav2vec)

print("\nИзвлечение Wav2Vec2 признаков для тестовой выборки...")
test_wav2vec = []
for audio in tqdm(test_audios):
    features = extract_wav2vec_features(audio, original_sr, wav2vec_model, processor)
    test_wav2vec.append(features)
test_wav2vec = np.array(test_wav2vec)

print(f"\nРазмерность признаков Wav2Vec2: {train_wav2vec.shape[1]}")
print(f"Тренировочная матрица: {train_wav2vec.shape}")
print(f"Тестовая матрица: {test_wav2vec.shape}")

In [ ]:
# Нормализация Wav2Vec2 признаков
scaler_wav2vec = StandardScaler()
train_wav2vec_scaled = scaler_wav2vec.fit_transform(train_wav2vec)
test_wav2vec_scaled = scaler_wav2vec.transform(test_wav2vec)

In [ ]:
# Визуализация Wav2Vec2 признаков
wav2vec_tsne, wav2vec_umap = visualize_features(train_wav2vec_scaled, train_labels, 'Wav2Vec2 признаки (frozen)')

In [ ]:
# Оценка классификаторов на Wav2Vec2 признаках
wav2vec_results, wav2vec_results_df = evaluate_classifiers(
    train_wav2vec_scaled, test_wav2vec_scaled,
    train_labels, test_labels,
    feature_name='Wav2Vec2 (frozen)'
)

## 6. Обучение классификационной головы (frozen backbone)

In [ ]:
class AudioClassifier(nn.Module):
    """
    Классификационная голова поверх Wav2Vec2.
    """
    def __init__(self, input_dim, num_classes=2, hidden_dim=256):
        super(AudioClassifier, self).__init__()
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim // 2, num_classes)
        )
    
    def forward(self, x):
        return self.classifier(x)

In [ ]:
# Подготовка данных для PyTorch
train_tensor = torch.FloatTensor(train_wav2vec_scaled)
train_labels_tensor = torch.LongTensor(train_labels)
test_tensor = torch.FloatTensor(test_wav2vec_scaled)
test_labels_tensor = torch.LongTensor(test_labels)

train_dataset = TensorDataset(train_tensor, train_labels_tensor)
test_dataset = TensorDataset(test_tensor, test_labels_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
def train_classifier(model, train_loader, test_loader, epochs=50, lr=0.001):
    """
    Обучает классификатор.
    """
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
    
    train_losses = []
    test_accuracies = []
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        avg_loss = total_loss / len(train_loader)
        train_losses.append(avg_loss)
        
        # Валидация
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for batch_x, batch_y in test_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                outputs = model(batch_x)
                _, predicted = torch.max(outputs.data, 1)
                total += batch_y.size(0)
                correct += (predicted == batch_y).sum().item()
        
        test_acc = correct / total
        test_accuracies.append(test_acc)
        scheduler.step(avg_loss)
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}, Test Acc: {test_acc:.4f}")
    
    return model, train_losses, test_accuracies

In [ ]:
# Обучение классификатора на замороженных признаках
input_dim = train_wav2vec.shape[1]
classifier_frozen = AudioClassifier(input_dim=input_dim, num_classes=2)

print("Обучение классификационной головы (frozen backbone)...")
classifier_frozen, losses_frozen, accs_frozen = train_classifier(
    classifier_frozen, train_loader, test_loader, epochs=50
)

In [ ]:
# Визуализация процесса обучения
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(losses_frozen)
axes[0].set_title('Training Loss (Frozen Backbone)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')

axes[1].plot(accs_frozen)
axes[1].set_title('Test Accuracy (Frozen Backbone)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')

plt.tight_layout()
plt.show()

print(f"\nФинальная точность классификатора (frozen): {accs_frozen[-1]:.4f}")

## 7. Fine-tuning: разморозка и дообучение Wav2Vec2

In [ ]:
class Wav2Vec2Classifier(nn.Module):
    """
    End-to-end модель: Wav2Vec2 + классификационная голова.
    """
    def __init__(self, wav2vec_model, num_classes=2, hidden_dim=256):
        super(Wav2Vec2Classifier, self).__init__()
        self.wav2vec = wav2vec_model
        self.hidden_size = self.wav2vec.config.hidden_size
        
        self.classifier = nn.Sequential(
            nn.Linear(self.hidden_size, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, num_classes)
        )
    
    def forward(self, input_values):
        outputs = self.wav2vec(input_values)
        # Усреднение по времени
        hidden_states = outputs.last_hidden_state.mean(dim=1)
        logits = self.classifier(hidden_states)
        return logits, hidden_states

In [ ]:
# Загрузка новой модели для fine-tuning
print("Загрузка модели для fine-tuning...")
wav2vec_finetune = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")

# Создание end-to-end модели
finetune_model = Wav2Vec2Classifier(wav2vec_finetune, num_classes=2)
finetune_model = finetune_model.to(device)

# Сначала заморозим Wav2Vec2 и обучим только голову
for param in finetune_model.wav2vec.parameters():
    param.requires_grad = False

print("Модель готова к fine-tuning")

In [ ]:
def prepare_audio_batch(audios, sr, target_sr=16000):
    """
    Подготавливает батч аудио для Wav2Vec2.
    """
    processed = []
    for audio in audios:
        if sr != target_sr:
            audio = librosa.resample(audio.astype(np.float32), orig_sr=sr, target_sr=target_sr)
        processed.append(audio)
    
    # Паддинг до одинаковой длины
    max_len = max(len(a) for a in processed)
    padded = np.zeros((len(processed), max_len))
    for i, a in enumerate(processed):
        padded[i, :len(a)] = a
    
    return torch.FloatTensor(padded)

In [ ]:
def train_finetune_model(model, train_audios, train_labels, test_audios, test_labels, 
                         sr, epochs=10, batch_size=8, lr=1e-5, unfreeze_after=5):
    """
    Обучает модель с fine-tuning.
    """
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    
    train_losses = []
    test_accuracies = []
    
    # Начинаем только с головы
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr*10)
    
    indices = np.arange(len(train_audios))
    
    for epoch in range(epochs):
        # Разморозка после определённой эпохи
        if epoch == unfreeze_after:
            print(f"\n*** Разморозка Wav2Vec2 на эпохе {epoch} ***\n")
            for param in model.wav2vec.parameters():
                param.requires_grad = True
            optimizer = optim.AdamW(model.parameters(), lr=lr)
        
        model.train()
        np.random.shuffle(indices)
        total_loss = 0
        num_batches = 0
        
        for i in range(0, len(indices), batch_size):
            batch_idx = indices[i:i+batch_size]
            batch_audios = [train_audios[j] for j in batch_idx]
            batch_labels = torch.LongTensor([train_labels[j] for j in batch_idx]).to(device)
            
            batch_input = prepare_audio_batch(batch_audios, sr).to(device)
            
            optimizer.zero_grad()
            logits, _ = model(batch_input)
            loss = criterion(logits, batch_labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            num_batches += 1
        
        avg_loss = total_loss / num_batches
        train_losses.append(avg_loss)
        
        # Тестирование
        model.eval()
        correct = 0
        total = 0
        
        with torch.no_grad():
            for i in range(0, len(test_audios), batch_size):
                batch_audios = test_audios[i:i+batch_size]
                batch_labels = torch.LongTensor(test_labels[i:i+batch_size]).to(device)
                
                batch_input = prepare_audio_batch(batch_audios, sr).to(device)
                logits, _ = model(batch_input)
                _, predicted = torch.max(logits.data, 1)
                
                total += batch_labels.size(0)
                correct += (predicted == batch_labels).sum().item()
        
        test_acc = correct / total
        test_accuracies.append(test_acc)
        
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}, Test Acc: {test_acc:.4f}")
    
    return model, train_losses, test_accuracies

In [ ]:
# Fine-tuning модели
print("Fine-tuning модели...")
finetune_model, losses_finetune, accs_finetune = train_finetune_model(
    finetune_model, train_audios, train_labels, test_audios, test_labels,
    sr=original_sr, epochs=15, batch_size=8, lr=1e-5, unfreeze_after=5
)

In [ ]:
# Визуализация fine-tuning
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(losses_finetune)
axes[0].axvline(x=5, color='r', linestyle='--', label='Unfreeze')
axes[0].set_title('Training Loss (Fine-tuning)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(accs_finetune)
axes[1].axvline(x=5, color='r', linestyle='--', label='Unfreeze')
axes[1].set_title('Test Accuracy (Fine-tuning)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nФинальная точность после fine-tuning: {accs_finetune[-1]:.4f}")

In [ ]:
# Извлечение признаков из дообученной модели
def extract_finetuned_features(audios, model, sr, batch_size=8):
    """
    Извлекает признаки из дообученной модели.
    """
    model.eval()
    all_features = []
    
    with torch.no_grad():
        for i in tqdm(range(0, len(audios), batch_size)):
            batch_audios = audios[i:i+batch_size]
            batch_input = prepare_audio_batch(batch_audios, sr).to(device)
            _, hidden_states = model(batch_input)
            all_features.append(hidden_states.cpu().numpy())
    
    return np.vstack(all_features)

print("Извлечение признаков из fine-tuned модели...")
train_finetuned = extract_finetuned_features(train_audios, finetune_model, original_sr)
test_finetuned = extract_finetuned_features(test_audios, finetune_model, original_sr)

# Нормализация
scaler_finetuned = StandardScaler()
train_finetuned_scaled = scaler_finetuned.fit_transform(train_finetuned)
test_finetuned_scaled = scaler_finetuned.transform(test_finetuned)

print(f"Размерность fine-tuned признаков: {train_finetuned.shape}")

In [ ]:
# Визуализация fine-tuned признаков
finetuned_tsne, finetuned_umap = visualize_features(
    train_finetuned_scaled, train_labels, 'Wav2Vec2 признаки (fine-tuned)'
)

In [ ]:
# Оценка классификаторов на fine-tuned признаках
finetuned_results, finetuned_results_df = evaluate_classifiers(
    train_finetuned_scaled, test_finetuned_scaled,
    train_labels, test_labels,
    feature_name='Wav2Vec2 (fine-tuned)'
)

## 8. Сравнение всех подходов

In [ ]:
# Сводная таблица результатов
all_results = {
    'MFCC': mfcc_results_df,
    'Wav2Vec2 (frozen)': wav2vec_results_df,
    'Wav2Vec2 (fine-tuned)': finetuned_results_df
}

# Добавляем результаты нейросети
nn_results = pd.DataFrame({
    'Признаки': ['Wav2Vec2 (frozen)', 'Wav2Vec2 (fine-tuned)'],
    'Модель': ['Neural Network', 'End-to-End Neural Network'],
    'Accuracy': [accs_frozen[-1], accs_finetune[-1]]
})

print("="*60)
print("СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ")
print("="*60)

for name, df in all_results.items():
    print(f"\n{name}:")
    print(df.to_string(index=False))

print(f"\nНейросетевые модели:")
print(nn_results.to_string(index=False))

In [ ]:
# Визуализация сравнения
fig, ax = plt.subplots(figsize=(12, 6))

# Собираем все результаты
comparison_data = []

for feature_name, df in all_results.items():
    for _, row in df.iterrows():
        comparison_data.append({
            'Признаки': feature_name,
            'Модель': row['Модель'],
            'F1-Score': row['F1-Score']
        })

# Добавляем нейросети
comparison_data.append({'Признаки': 'Wav2Vec2 (frozen)', 'Модель': 'Neural Network', 'F1-Score': accs_frozen[-1]})
comparison_data.append({'Признаки': 'Wav2Vec2 (fine-tuned)', 'Модель': 'End-to-End NN', 'F1-Score': accs_finetune[-1]})

comparison_df = pd.DataFrame(comparison_data)

# Построение графика
sns.barplot(data=comparison_df, x='Модель', y='F1-Score', hue='Признаки', ax=ax)
ax.set_title('Сравнение F1-Score для разных признаков и моделей')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.legend(title='Признаки', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 9. Анализ лучшего признакового пространства: ближайшие и дальние аудиозаписи

In [ ]:
# Выбираем лучшее признаковое пространство на основе результатов
# Сравниваем лучшие F1-Score для каждого признакового пространства

best_scores = {
    'MFCC': mfcc_results_df['F1-Score'].max(),
    'Wav2Vec2 (frozen)': max(wav2vec_results_df['F1-Score'].max(), accs_frozen[-1]),
    'Wav2Vec2 (fine-tuned)': max(finetuned_results_df['F1-Score'].max(), accs_finetune[-1])
}

best_feature_space = max(best_scores, key=best_scores.get)
print(f"Лучшее признаковое пространство: {best_feature_space}")
print(f"Лучший F1-Score: {best_scores[best_feature_space]:.4f}")

# Выбираем соответствующие признаки
feature_spaces = {
    'MFCC': (train_mfcc_scaled, train_audios),
    'Wav2Vec2 (frozen)': (train_wav2vec_scaled, train_audios),
    'Wav2Vec2 (fine-tuned)': (train_finetuned_scaled, train_audios)
}

best_features, best_audios = feature_spaces[best_feature_space]

In [ ]:
from scipy.spatial.distance import pdist, squareform

# Вычисление матрицы расстояний
print("Вычисление матрицы расстояний...")
distances = squareform(pdist(best_features, metric='euclidean'))

# Находим пару с минимальным расстоянием (исключая диагональ)
np.fill_diagonal(distances, np.inf)
min_idx = np.unravel_index(np.argmin(distances), distances.shape)
closest_pair = min_idx

# Находим пару с максимальным расстоянием
np.fill_diagonal(distances, -np.inf)
max_idx = np.unravel_index(np.argmax(distances), distances.shape)
farthest_pair = max_idx

print(f"\nБлижайшие аудиозаписи: индексы {closest_pair[0]} и {closest_pair[1]}")
print(f"Расстояние: {distances[closest_pair[0], closest_pair[1]]:.4f}")
print(f"Классы: {train_labels[closest_pair[0]]} и {train_labels[closest_pair[1]]}")

np.fill_diagonal(distances, np.inf)  # Восстанавливаем для корректного отображения
print(f"\nНаиболее дальние аудиозаписи: индексы {farthest_pair[0]} и {farthest_pair[1]}")
print(f"Расстояние: {distances[farthest_pair[0], farthest_pair[1]]:.4f}")
print(f"Классы: {train_labels[farthest_pair[0]]} и {train_labels[farthest_pair[1]]}")

In [ ]:
def plot_spectrogram(audio, sr, title, ax):
    """
    Визуализирует спектрограмму аудио.
    """
    # Вычисление мел-спектрограммы
    S = librosa.feature.melspectrogram(y=audio.astype(np.float32), sr=sr, n_mels=128)
    S_dB = librosa.power_to_db(S, ref=np.max)
    
    img = librosa.display.specshow(S_dB, x_axis='time', y_axis='mel', sr=sr, ax=ax)
    ax.set_title(title)
    return img

In [ ]:
# Визуализация спектрограмм ближайших аудиозаписей
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Ближайшие
idx1, idx2 = closest_pair
plot_spectrogram(best_audios[idx1], original_sr, 
                 f'Ближайшая пара - Аудио 1 (idx={idx1}, класс={train_labels[idx1]})', axes[0, 0])
plot_spectrogram(best_audios[idx2], original_sr, 
                 f'Ближайшая пара - Аудио 2 (idx={idx2}, класс={train_labels[idx2]})', axes[0, 1])

# Дальние
idx3, idx4 = farthest_pair
plot_spectrogram(best_audios[idx3], original_sr, 
                 f'Дальняя пара - Аудио 1 (idx={idx3}, класс={train_labels[idx3]})', axes[1, 0])
img = plot_spectrogram(best_audios[idx4], original_sr, 
                       f'Дальняя пара - Аудио 2 (idx={idx4}, класс={train_labels[idx4]})', axes[1, 1])

# Общий colorbar
fig.colorbar(img, ax=axes, format='%+2.0f dB', label='Мощность (dB)')

plt.suptitle(f'Спектрограммы для {best_feature_space}', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Визуализация положения этих точек в признаковом пространстве
fig, ax = plt.subplots(figsize=(10, 8))

# Используем UMAP для визуализации
umap = UMAP(n_components=2, random_state=42)
features_2d = umap.fit_transform(best_features)

# Все точки
scatter = ax.scatter(features_2d[:, 0], features_2d[:, 1], 
                     c=train_labels, cmap='coolwarm', alpha=0.5, s=30)

# Выделяем ближайшую пару
ax.scatter(features_2d[closest_pair, 0], features_2d[closest_pair, 1], 
           c='green', s=200, marker='*', edgecolors='black', linewidths=2,
           label='Ближайшая пара', zorder=5)
ax.plot(features_2d[closest_pair, 0], features_2d[closest_pair, 1], 
        'g--', linewidth=2, alpha=0.7)

# Выделяем дальнюю пару
ax.scatter(features_2d[list(farthest_pair), 0], features_2d[list(farthest_pair), 1], 
           c='purple', s=200, marker='D', edgecolors='black', linewidths=2,
           label='Дальняя пара', zorder=5)
ax.plot(features_2d[list(farthest_pair), 0], features_2d[list(farthest_pair), 1], 
        'purple', linestyle='--', linewidth=2, alpha=0.7)

ax.set_title(f'Визуализация ближайших и дальних пар в {best_feature_space}')
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
plt.colorbar(scatter, label='Класс')
ax.legend()
plt.tight_layout()
plt.show()

## 10. Выводы

### Результаты экспериментов:

1. **MFCC признаки** - классические признаки, извлечённые с помощью librosa. Включают MFCC, дельта и дельта-дельта коэффициенты, агрегированные через mean и std.

2. **Wav2Vec2 (frozen)** - признаки из предобученной модели без дообучения. Модель работает с сырым аудио на частоте 16kHz.

3. **Wav2Vec2 (fine-tuned)** - признаки после дообучения всей модели на нашем датасете.

### Наблюдения:

- Fine-tuning обычно улучшает качество признаков для конкретной задачи
- Классические модели (Random Forest, SVM) хорошо работают даже на признаках из глубоких моделей
- Визуализация показывает, как классы разделяются в разных признаковых пространствах
- Ближайшие аудиозаписи часто принадлежат одному классу, дальние - разным

In [ ]:
# Финальная сводка
print("="*60)
print("ФИНАЛЬНАЯ СВОДКА")
print("="*60)
print(f"\nЛучшее признаковое пространство: {best_feature_space}")
print(f"Лучший F1-Score: {best_scores[best_feature_space]:.4f}")
print(f"\nСравнение лучших результатов по признаковым пространствам:")
for name, score in sorted(best_scores.items(), key=lambda x: -x[1]):
    print(f"  {name}: {score:.4f}")